[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/Filtered_basline_sweep%20%281%29.ipynb)


In [ ]:
# Colab / local repository setup
# Run this cell first when opening the notebook in Google Colab. It clones the
# repository, installs the pinned requirements, and makes data/ plus src/ imports
# available from the same execution context used by the local notebooks.
from pathlib import Path
import os
import subprocess
import sys


def _running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules


def _run(command, cwd=None):
    print("$", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)


REPO_URL = os.environ.get("ASTROMODEL_REPO_URL", "https://github.com/doxav/astromodel_proving.git")
REPO_BRANCH = os.environ.get("ASTROMODEL_REPO_BRANCH", "main")
PROJECT_DIRNAME = os.environ.get("ASTROMODEL_PROJECT_DIRNAME", "astromodel_proving")

if _running_in_colab():
    project_root = Path("/content") / PROJECT_DIRNAME
    if not project_root.exists():
        clone_cmd = [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            REPO_BRANCH,
            REPO_URL,
            str(project_root),
        ]
        try:
            _run(clone_cmd)
        except subprocess.CalledProcessError:
            # Some forks/default branches may not be named like REPO_BRANCH.
            # Retry without an explicit branch before surfacing the clone error.
            _run(["git", "clone", "--depth", "1", REPO_URL, str(project_root)])
    os.chdir(project_root)
    requirements = project_root / "requirements.txt"
    if requirements.exists():
        _run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
else:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    project_root = next(
        (
            candidate
            for candidate in candidates
            if (candidate / "src").is_dir() and (candidate / "data").is_dir()
        ),
        current,
    )
    os.chdir(project_root)

os.environ["ASTROMODEL_PROJECT_ROOT"] = str(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"ASTROMODEL_PROJECT_ROOT={project_root}")
print(f"Working directory={Path.cwd()}")
print(f"data exists={(project_root / 'data').exists()}, src exists={(project_root / 'src').exists()}")


GPT-PROMPT-RELATED TO THIS SCRIPT:
https://chatgpt.com/share/69b575cf-6d34-800f-9022-a1630c1fc24f
https://chatgpt.com/share/69b156c4-e560-800f-bdf0-5cfbe8b1acb3
https://chatgpt.com/share/69e7308b-60f8-832a-ad13-cf2469800f0d
K release and speed calculation
:https://chatgpt.com/share/69e7315b-2424-832c-aa7d-68c409b27ba8


In [ ]:
"""
Colab-ready post-optimization analysis for the astrocyte K-buffering model.

What this script does
---------------------
1) Loads one Optuna SQLite database and one ATF trace file.
2) Infers experiment metadata from the study name unless you override it.
3) Extracts the top completed, non-penalty trials from the DB.
4) Reconstructs full parameter sets by merging each trial with the enqueued default
   parameters stored in the DB (`fixed_params`).
5) Re-simulates those trials with the original model and overlays the interpolated
   simulated traces with the experimental trace.
6) Extracts the same Vm features used in ATF_Analysis.
7) Filters trials against the experimental threshold CSV to define "good enough" fits.
8) Uses accepted fits as a baseline ensemble and performs one-at-a-time parameter
   sweeps for gkir, d, Pk, Zth, Zs, and gs.
9) Saves tables and plots for visual inspection and downstream analysis.

The script is intentionally written as a single file for straightforward Colab use.

Typical Colab usage
-------------------
- Upload this script plus:
    * the Optuna .db file
    * the relevant ATF file
    * threshold_for_good_enough_fits.csv
- Edit the CONFIG block below.
- Run:
      !python astro_post_optuna_analysis.py

Notes
-----
- By default, the acceptance band uses pooled experimental min/max for the matching
  sweep index. You can switch to q1-q3 or CI95 in the CONFIG block.
- The script keeps the original model structure and the original ATF preprocessing
  logic from the optimization notebook.
- Parameter sweeps are one-at-a-time around each accepted baseline.
"""

from __future__ import annotations

import csv
import json
import math
import os
import re
import sqlite3
import subprocess
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy import stats

In [ ]:

# =========================
# CONFIG
# =========================
DB_PATH = "/content/CONTROL_175nA.db"
ATF_PATH = "/content/CONTROL_TRACES.atf"
THRESHOLD_CSV_PATH = "/content/threshold_for_good_enough_fits.csv"
OUT_DIR = "/content/astro_post_optuna_outputs"

# Optional override. If left as None, they are inferred from the study name in the DB.
STUDY_NAME: Optional[str] = None
EXPERIMENT_TYPE: Optional[str] = None        # CONTROL | MFA | BARIUM
CURRENT_NA: Optional[int] = None             # 50 | 75 | 100 | 125 | 150 | 175
TARGET_MEAN_MODE: Optional[str] = None       # default | centered | centered_scaled

# Top trials
TOP_N_REQUESTED = 300
PENALTY_VALUE = 1.0e7
INCLUDE_PENALTY_TRIALS = False

# Feature acceptance
THRESHOLD_GROUP = "pooled"                   # pooled | T | VH
THRESHOLD_MODE = "min_max"                   # min_max | q1_q3 | ci95
FEATURES_FOR_FILTER = [
    "peak_depolarization_mV",
    "rise_slope_mV_per_s",
    "rise_tau_s",
    "plateau_slope_mV_per_s",
    "decay_slope_mV_per_s",
    "decay_tau_s",
    "undershoot_magnitude_mV",
    "return_slope_mV_per_s",
]

# Softer acceptance controls
# - FEATURES_FOR_FILTER defines which features are considered.
# - FEATURE_ACCEPTANCE_MODE controls how many of those selected features must pass.
# - SOFT_THRESHOLD_PAD_FRAC widens each accepted band by a fraction of its width.
FEATURE_ACCEPTANCE_MODE = "fraction"         # all | any | fraction | max_failed
MIN_PASS_FRACTION = 0.75                     # used when FEATURE_ACCEPTANCE_MODE == "fraction"
MAX_FAILED_FEATURES = 2                      # used when FEATURE_ACCEPTANCE_MODE == "max_failed"
SOFT_THRESHOLD_PAD_FRAC = 0.10               # 0.10 -> widen each band by 10% on both sides
SOFT_THRESHOLD_PAD_ABS = 0.0                 # absolute padding in feature units; added via max(abs, frac*width)

# Backward-compatible switch. If you set this to True manually, it overrides the soft mode and requires all selected features.
REQUIRE_ALL_FEATURES_IN_RANGE = False

# Feature extraction details (kept aligned with ATF_Analysis)
SMOOTHING_WINDOW_S = 0.025
RETURN_SLOPE_FINAL_VALUE_MODE = "last_200ms_mean"   # last_point | last_200ms_mean

# Parameter sweeps around accepted baselines (fold change relative to each accepted baseline)
PARAM_SWEEP_FOLDS = {
    "gki": [0.5, 0.75, 1.0, 1.25, 1.5, 2.0],
    "d":   [0.5, 0.75, 1.0, 1.25, 1.5, 2.0],
    "pk":  [0.5, 0.75, 1.0, 1.25, 1.5, 2.0],
    "zth": [0.5, 0.75, 1.0, 1.25, 1.5],
    "zs":  [0.5, 0.75, 1.0, 1.25, 1.5],
    "gs":  [0.5, 0.75, 1.0, 1.25, 1.5, 2.0],
}

# If accepted fits are too many for perturbation sweeps, you can cap them here.
MAX_ACCEPTED_FOR_SWEEPS: Optional[int] = None # e.g 20

# Original simulation settings from the optimization code
SIM_DT_MS = 0.1
Z0 = [-89.0, 0.0, 0.0, 0.0]

# Plotting
TRACE_ALPHA_TOP = 0.18
TRACE_ALPHA_ACCEPTED = 0.30
BEST_TRACE_LINEWIDTH = 1.8
EXPERIMENT_LINEWIDTH = 2.2


In [ ]:
# =========================
# Dependency helper
# =========================
def ensure_optuna() -> None:
    try:
        import optuna  # noqa: F401
    except Exception:
        print("Installing optuna for DB reading...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna>=3.5,<4"])


ensure_optuna()
import optuna  # noqa: E402

Installing optuna for DB reading...


In [ ]:
# =========================
# Constants copied from the optimization workflow
# =========================
CURRENT_DICT_COLUMNS = {"50": 1, "75": 2, "100": 3, "125": 4, "150": 5, "175": 6}
CURRENT_DICT_K_BATH_VALUES = {
    "50": [4.8, 6.4, 4.8],
    "75": [4.8, 7.23, 4.8],
    "100": [4.8, 8.2, 4.8],
    "125": [4.8, 9.5, 4.8],
    "150": [4.8, 10.1, 4.8],
    "175": [4.8, 10.5, 4.8],
}
EXPERIMENT_K_BATH_TIME = {
    "MFA": [0.0, 21140.0, 41140.0],
    "BARIUM": [0.0, 21140.0, 41140.0],
    "CONTROL": [0.0, 11173.0, 31173.0],
}
EXPERIMENT_CUTS = {
    "MFA": [30, 0.2],
    "BARIUM": [30, 0.2],
    "CONTROL": [0, 0.1],
}

ORDERED_PARAM_KEYS = [
    "eps", "eps_middle", "gki", "pk", "gs", "gt", "ca", "d", "w_a",
    "wo", "wo_middle", "K_bath_value_middle", "gl_a", "Va_s", "Va_l",
    "switching_function", "zth", "zs", "hill_coefficient", "K_d",
]

FEATURE_ORDER = [
    "peak_depolarization_mV",
    "rise_slope_mV_per_s",
    "rise_tau_s",
    "plateau_slope_mV_per_s",
    "decay_slope_mV_per_s",
    "decay_tau_s",
    "undershoot_magnitude_mV",
    "return_slope_mV_per_s",
]



In [ ]:
# =========================
# Data classes
# =========================
@dataclass
class RunConfig:
    db_path: Path
    atf_path: Path
    threshold_csv_path: Path
    out_dir: Path
    study_name: Optional[str]
    experiment_type: Optional[str]
    current_na: Optional[int]
    target_mean_mode: Optional[str]


@dataclass
class ExperimentContext:
    experiment_type: str
    current_na: int
    target_mean_mode: str
    study_name: str
    current_column_id: int
    sweep_id: int
    cut_after_index: int
    cut_before_ratio: float
    exp_times_ms_stable: np.ndarray
    exp_trace_stable: np.ndarray
    sim_time_ms: np.ndarray
    sim_time_ms_stable: np.ndarray
    stable_index_simulation: int
    fixed_params: Dict[str, object]
    threshold_df: pd.DataFrame
    feature_onset_s: float
    feature_offset_s: float


In [ ]:
# =========================
# Utility functions
# =========================
def make_output_dirs(base_dir: Path) -> Dict[str, Path]:
    paths = {
        "base": base_dir,
        "plots": base_dir / "plots",
        "tables": base_dir / "tables",
        "sweeps": base_dir / "parameter_sweeps",
        "sweep_plots": base_dir / "parameter_sweeps" / "plots",
    }
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths


def infer_study_name_from_db(db_path: Path) -> str:
    storage = f"sqlite:///{db_path}"
    summaries = optuna.get_all_study_summaries(storage=storage)
    if not summaries:
        raise ValueError(f"No Optuna studies found in {db_path}")
    return summaries[0].study_name


def infer_metadata_from_study_name(study_name: str) -> Tuple[str, int, str]:
    parts = study_name.split("_")
    if len(parts) < 3:
        raise ValueError(f"Could not parse study metadata from study name: {study_name}")

    experiment_type = parts[0]
    current_na = int(parts[1].replace("nA", ""))

    # target_mean_mode can be "centered_scaled", so detect that case explicitly
    if len(parts) >= 4 and parts[2] == "centered" and parts[3] == "scaled":
        target_mean_mode = "centered_scaled"
    else:
        target_mean_mode = parts[2]

    return experiment_type, current_na, target_mean_mode


def load_study(db_path: Path, study_name: Optional[str] = None):
    storage = f"sqlite:///{db_path}"
    if study_name is None:
        study_name = infer_study_name_from_db(db_path)
    return optuna.load_study(study_name=study_name, storage=storage)


def load_fixed_params_from_db(db_path: Path) -> Dict[str, object]:
    query = """
    SELECT value_json
    FROM trial_system_attributes
    WHERE key = 'fixed_params'
    ORDER BY trial_id ASC
    LIMIT 1
    """
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query(query, conn)
    if df.empty:
        raise ValueError(
            "No 'fixed_params' attribute found in the DB. "
            "This script expects the study to contain at least one enqueued trial."
        )
    fixed = json.loads(df.loc[0, "value_json"])
    return normalize_flat_params(fixed)


def normalize_flat_params(params: Dict[str, object]) -> Dict[str, object]:
    p = dict(params)

    # normalize a few alternative spellings that sometimes appear in notebooks
    if "Zth" in p and "zth" not in p:
        p["zth"] = p["Zth"]
    if "Zs" in p and "zs" not in p:
        p["zs"] = p["Zs"]
    if "Z_th" in p and "zth" not in p:
        p["zth"] = p["Z_th"]
    if "Z_s" in p and "zs" not in p:
        p["zs"] = p["Z_s"]
    if "w_o_middle" in p and "wo_middle" not in p:
        p["wo_middle"] = p["w_o_middle"]

    # fill standard defaults used by the original code when missing
    p.setdefault("wo_middle", 1.0)
    p.setdefault("eps_middle", 1.0)
    p.setdefault("w_a", 2000.0)
    p.setdefault("switching_function", "sigmoid")
    return p


def normalize_trace_for_target_mode(x: np.ndarray, target_mean_mode: str) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if target_mean_mode == "centered":
        return x - np.mean(x)
    if target_mean_mode == "centered_scaled":
        denom = np.max(x) - np.min(x)
        if denom == 0:
            return x - np.mean(x)
        return (x - np.mean(x)) / denom
    return x.copy()


def downsample_array_median_fast(sols: np.ndarray, num_samples: int) -> np.ndarray:
    sols = np.asarray(sols, dtype=float)
    points_per_segment = len(sols) // num_samples
    if points_per_segment < 1:
        raise ValueError(
            f"Cannot downsample array of length {len(sols)} to {num_samples} samples."
        )
    trimmed_length = points_per_segment * num_samples
    trimmed_sols = sols[:trimmed_length]
    reshaped_sols = trimmed_sols.reshape(num_samples, points_per_segment)
    return np.median(reshaped_sols, axis=1)


def compute_loss(z: np.ndarray, target: np.ndarray, loss_type: str = "COMBINED",
                 delta_huber: float = 1.0, gradient_loss_weight: float = 20.0) -> float:
    z = np.asarray(z, dtype=float)
    target = np.asarray(target, dtype=float)

    if loss_type == "L2":
        return float(np.sum((z - target) ** 2))
    if loss_type == "L1":
        return float(np.sum(np.abs(z - target)))
    if loss_type == "HUBER":
        abs_diff = np.abs(z - target)
        is_small_error = abs_diff <= delta_huber
        squared_loss = 0.5 * (abs_diff ** 2)
        linear_loss = delta_huber * (abs_diff - 0.5 * delta_huber)
        return float(np.sum(np.where(is_small_error, squared_loss, linear_loss)))
    if loss_type == "LOG_COSH":
        return float(np.sum(np.log(np.cosh(z - target))))
    if loss_type == "COMBINED":
        l2_loss = np.sum((z - target) ** 2)
        z_grad = np.gradient(z)
        target_grad = np.gradient(target)
        gradient_loss = gradient_loss_weight * np.sum(np.abs(z_grad - target_grad))
        return float(l2_loss + gradient_loss)
    raise ValueError(f"Unknown loss type: {loss_type}")

In [ ]:
# =========================
# Model
# =========================
def model(z: Sequence[float], t: float, paramdict: Dict[str, Dict[str, object]]) -> List[float]:
    Cm_a = paramdict["Astrocyte"]["Cm_a"]
    g_kir = paramdict["Astrocyte"]["g_kir"]
    A = paramdict["Astrocyte"]["A"]
    g_k_a = paramdict["Astrocyte"]["g_k_a"]
    gl_a = paramdict["Astrocyte"]["gl_a"]
    w_a = paramdict["Astrocyte"]["w_a"]
    K_a0 = paramdict["Astrocyte"]["K_a0"]
    Sig_a = paramdict["Astrocyte"]["Sig_a"]
    gama_t = paramdict["Astrocyte"]["gama_t"]
    gama_s = paramdict["Astrocyte"]["gama_s"]
    Z_th = paramdict["Astrocyte"]["Z_th"]
    Z_s = paramdict["Astrocyte"]["Z_s"]
    Va_0 = paramdict["Astrocyte"]["Va_0"]
    Va_s = paramdict["Astrocyte"]["Va_s"]
    Va_l = paramdict["Astrocyte"]["Va_l"]
    P_k = paramdict["Astrocyte"]["P_k"]
    d_gap = paramdict["Astrocyte"]["d_gap"]
    F = paramdict["Astrocyte"]["F"]
    R = paramdict["Astrocyte"]["R"]
    T = paramdict["Astrocyte"]["T"]

    K_o0 = paramdict["external"]["K_o0"]
    w_o = paramdict["external"]["w_o"]
    epsilon = paramdict["external"]["epsilon"]

    idx = np.where(paramdict["external"]["K_bath"]["time"] <= t)[0][-1]
    K_bath = paramdict["external"]["K_bath"]["value"][idx]

    switching_function = paramdict["Astrocyte"].get("switching_function", "sigmoid")

    if "epsilon_middle" in paramdict["external"] and idx == 1:
        epsilon = epsilon * paramdict["external"]["epsilon_middle"]
    if "w_o_middle" in paramdict["external"] and idx == 1:
        w_o = w_o * paramdict["external"]["w_o_middle"]

    Va = z[0]
    DK_a_t = z[1]
    K_a_s = z[2]
    Kg = z[3]

    DK_a = DK_a_t + K_a_s
    K_a = K_a0 + DK_a
    DK_o_a = -(w_a / w_o) * DK_a_t
    K_o = K_o0 + DK_o_a + Kg

    K_ratio = K_o / K_a
    if K_ratio <= 0:
        K_ratio = 1e-8
    E_k_a = 25.7 * np.log(K_ratio)
    I_k_a = g_k_a * (Va - E_k_a)
    I_Kir = g_kir * np.sqrt(np.abs(K_o)) * (Va - E_k_a) * (1 / (1 + np.exp((Va - E_k_a) / 19.2)))
    PH_a = 0.04 * (Va - Va_s)
    P_kgap = d_gap * P_k
    exp_neg_PH_a = np.exp(-PH_a)
    denominator = -1 + np.exp(-PH_a)
    if denominator == 0:
        denominator = 1e-8
    I_kgap = P_kgap * F * PH_a * (1 / denominator) * ((K_a * exp_neg_PH_a) - K_a0)
    I_l_a = gl_a * (Va - Va_l)

    if switching_function == "sigmoid":
        Th_s = DK_a / (1 + np.exp((Z_th - DK_a_t) * Z_s))
    elif switching_function == "tanh":
        Th_s = DK_a * (0.5 * (1 + np.tanh((DK_a_t - Z_th) * Z_s)))
    elif switching_function == "hill":
        n = paramdict["Astrocyte"].get("hill_coefficient", 2)
        K_d = paramdict["Astrocyte"].get("K_d", 1)
        Th_s = DK_a * ((DK_a_t ** n) / (K_d ** n + DK_a_t ** n))
    else:
        raise ValueError(f"Unknown switching function type: {switching_function}")

    dVa = (-1.0 / Cm_a) * (I_Kir + I_k_a + I_l_a + I_kgap)
    dDK_a_t = -(gama_t * Sig_a / (w_a * F)) * (I_Kir + I_k_a)
    dK_a_s = -Th_s * (gama_s * Sig_a / (w_a * F)) * I_kgap
    dKg = epsilon * (K_bath - K_o)

    return [dVa, dDK_a_t, dK_a_s, dKg]


def build_paramdict(experiment_type: str, current_na: int, flat_params: Dict[str, object]) -> Dict[str, Dict[str, object]]:
    p = normalize_flat_params(flat_params)
    current_key = str(current_na)

    paramdict = {
        "Astrocyte": {
            "Cm_a": float(p.get("ca", 400.0)),
            "g_kir": float(p.get("gki", 1.0)),
            "g_k_a": 0.0,
            "w_a": float(p.get("w_a", 2000.0)),
            "P_k": float(p.get("pk", 3e-5)),
            "A": 1.0,
            "gl_a": float(p.get("gl_a", 0.01)),
            "Va_l": float(p.get("Va_l", -70.0)),
            "Va_s": float(p.get("Va_s", -90.0)),
            "d_gap": float(p.get("d", 1.0)),
            "Va_0": -89.0,
            "Sig_a": 1600.0,
            "K_a0": 135.0,
            "F": 96485.0,
            "R": 8.314,
            "T": 298.0,
            "gama_t": float(p.get("gt", 6.0)),
            "gama_s": float(p.get("gs", 6.5)),
            "Z_s": float(p.get("zs", 0.05)) if p.get("zs", None) is not None else None,
            "Z_th": float(p.get("zth", 0.2)) if p.get("zth", None) is not None else None,
            "switching_function": str(p.get("switching_function", "sigmoid")),
        },
        "external": {
            "K_o0": 4.8,
            "w_o": float(p.get("wo", 1500.0)),
            "w_o_middle": float(p.get("wo_middle", 1.0)),
            "epsilon": float(p.get("eps", 1e-3)),
            "epsilon_middle": float(p.get("eps_middle", 1.0)),
            "K_bath": {
                "time": np.array(EXPERIMENT_K_BATH_TIME[experiment_type], dtype=float),
                "value": CURRENT_DICT_K_BATH_VALUES[current_key].copy(),
            },
        },
    }

    paramdict["external"]["K_bath"]["value"][1] = float(
        p.get("K_bath_value_middle", paramdict["external"]["K_bath"]["value"][1])
    )

    if paramdict["Astrocyte"]["switching_function"] == "hill":
        paramdict["Astrocyte"]["hill_coefficient"] = float(p.get("hill_coefficient", 2.0))
        paramdict["Astrocyte"]["K_d"] = float(p.get("K_d", 1.0))

    return paramdict

In [ ]:
# =========================
# ATF loading and preprocessing
# =========================
def load_atf_file_numeric(file_path: Path) -> np.ndarray:
    with open(file_path, "r") as file:
        lines = file.readlines()

    data_start_index = next(i for i, line in enumerate(lines) if "Signals" in line) + 1
    tabular_data = lines[data_start_index:]

    processed_data = []
    for row in tabular_data:
        if row.strip():
            values = row.replace('"', "").split("\t")
            try:
                processed_data.append([float(value) for value in values])
            except ValueError:
                continue

    data_array = np.array(processed_data, dtype=float)
    data_array[:, 0] *= 1000.0  # seconds -> ms
    return data_array


def load_experimental_trace(atf_path: Path, experiment_type: str, current_na: int,
                            target_mean_mode: str) -> Tuple[np.ndarray, np.ndarray]:
    data = load_atf_file_numeric(atf_path)
    current_column_id = CURRENT_DICT_COLUMNS[str(current_na)]
    cut_after_index = EXPERIMENT_CUTS[experiment_type][0]
    cut_before_ratio = EXPERIMENT_CUTS[experiment_type][1]

    index_to_cut = data.shape[0] - cut_after_index
    data_cut = data[:index_to_cut]

    # The original optimization notebook wrote the cut array to CSV with no header,
    # then reopened it and skipped the first row as if it were a header.
    # We reproduce that behavior here for consistency with the stored objective values.
    data_cut = data_cut[1:]

    times_ms = data_cut[:, 0]
    trace = data_cut[:, current_column_id]

    stable_index_empirical = int(round(len(times_ms) * cut_before_ratio))
    if stable_index_empirical >= len(times_ms):
        raise ValueError("stable_index_empirical is beyond the array length")

    exp_times_ms_stable = times_ms[stable_index_empirical:]
    exp_trace_stable = trace[stable_index_empirical:].copy()

    # Preserve the original cleanup from the notebook
    if experiment_type == "MFA" and current_na == 100:
        outlier_mask = (exp_times_ms_stable >= 22649) & (exp_times_ms_stable <= 22650)
        if np.any(outlier_mask):
            outlier_start_index = np.where(outlier_mask)[0][0] - 1
            exp_trace_stable[outlier_mask] = exp_trace_stable[outlier_start_index]

    exp_trace_stable = normalize_trace_for_target_mode(exp_trace_stable, target_mean_mode)
    return exp_times_ms_stable, exp_trace_stable

In [ ]:
# =========================
# Feature extraction (ported from ATF_Analysis)
# =========================
def moving_average(x: np.ndarray, window_pts: int) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    n = max(3, int(window_pts))
    if n % 2 == 0:
        n += 1
    kernel = np.ones(n, dtype=float) / n
    return np.convolve(x, kernel, mode="same")


def detect_step_edges(t: np.ndarray, x: np.ndarray, response_only: bool = False) -> Tuple[float, float]:
    t = np.asarray(t, dtype=float)
    x = np.asarray(x, dtype=float)

    dt = float(np.median(np.diff(t)))
    x_smooth = moving_average(x, max(5, int(round(0.05 / dt))))
    dx = np.diff(x_smooth)
    duration = float(t[-1] - t[0])

    if response_only:
        onset_mask = (t[1:] >= t[0] + 0.10 * duration) & (t[1:] <= t[0] + 0.60 * duration)
    else:
        onset_mask = (t[1:] >= t[0] + 0.02 * duration) & (t[1:] <= t[0] + 0.70 * duration)

    onset_candidates = np.where(onset_mask)[0]
    onset_idx = onset_candidates[np.argmax(dx[onset_mask])] if len(onset_candidates) else int(np.argmax(dx))

    min_gap = max(2.0, 0.05 * duration)
    offset_mask = (t[1:] >= t[onset_idx + 1] + min_gap) & (t[1:] <= t[-1] - min_gap)
    offset_candidates = np.where(offset_mask)[0]
    if len(offset_candidates):
        offset_idx = offset_candidates[np.argmin(dx[offset_mask])]
    else:
        offset_idx = onset_idx + 1 + int(np.argmin(dx[onset_idx + 1:]))

    return float(t[onset_idx + 1]), float(t[offset_idx + 1])


def first_crossing_time(t: np.ndarray, x: np.ndarray, threshold: float,
                        start_idx: int, end_idx: Optional[int] = None,
                        direction: str = "up") -> float:
    if end_idx is None:
        end_idx = len(x) - 1
    seg = x[start_idx:end_idx + 1]
    ts = t[start_idx:end_idx + 1]
    if len(seg) < 2:
        return np.nan

    if direction == "up":
        idx = np.where(seg >= threshold)[0]
    else:
        idx = np.where(seg <= threshold)[0]

    if len(idx) == 0:
        return np.nan
    return float(ts[idx[0]])


def line_slope(t: np.ndarray, y: np.ndarray) -> float:
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(t) < 2 or np.allclose(t, t[0]):
        return np.nan
    return float(np.polyfit(t, y, 1)[0])


def extract_features_from_trace(
    t_s: np.ndarray,
    v_raw: np.ndarray,
    onset_s: Optional[float] = None,
    offset_s: Optional[float] = None,
    smoothing_window_s: float = SMOOTHING_WINDOW_S,
    final_value_mode: str = RETURN_SLOPE_FINAL_VALUE_MODE,
) -> Dict[str, float]:
    t = np.asarray(t_s, dtype=float)
    v_raw = np.asarray(v_raw, dtype=float)
    dt = float(np.median(np.diff(t)))

    if onset_s is None or offset_s is None:
        onset_s, offset_s = detect_step_edges(t, v_raw, response_only=True)

    v = moving_average(v_raw, max(5, int(round(smoothing_window_s / dt))))

    baseline_mask = (t >= max(t[0], onset_s - 5.0)) & (t < onset_s - 1.0)
    if not np.any(baseline_mask):
        raise ValueError("Baseline window is empty. Check trace duration or onset time.")

    baseline = float(np.median(v_raw[baseline_mask]))

    stim_mask = (t >= onset_s) & (t <= offset_s)
    post_mask = (t >= offset_s) & (t <= min(t[-1], offset_s + 10.0))
    if not np.any(stim_mask):
        raise ValueError("Stimulus window is empty.")
    if not np.any(post_mask):
        raise ValueError("Post-stimulus window is empty.")

    stim_t = t[stim_mask]
    stim_v = v[stim_mask]
    post_t = t[post_mask]
    post_v = v[post_mask]

    peak_rel_idx = int(np.argmax(stim_v))
    peak_t = float(stim_t[peak_rel_idx])
    peak_v = float(stim_v[peak_rel_idx])
    peak_dep = peak_v - baseline

    onset_idx = np.searchsorted(t, onset_s)
    peak_idx = np.searchsorted(t, peak_t)

    thr20 = baseline + 0.2 * peak_dep
    thr63 = baseline + 0.632 * peak_dep
    thr80 = baseline + 0.8 * peak_dep

    t20 = first_crossing_time(t, v, thr20, onset_idx, peak_idx, direction="up")
    t63 = first_crossing_time(t, v, thr63, onset_idx, peak_idx, direction="up")
    t80 = first_crossing_time(t, v, thr80, onset_idx, peak_idx, direction="up")

    rise_slope = np.nan
    if np.isfinite(t20) and np.isfinite(t80) and t80 > t20:
        rise_slope = (0.6 * peak_dep) / (t80 - t20)

    rise_tau = t63 - onset_s if np.isfinite(t63) else np.nan

    plateau_start = onset_s + 3.0
    plateau_end = offset_s - 1.0
    if plateau_end <= plateau_start:
        plateau_start = onset_s + 1.0
        plateau_end = offset_s - 0.5

    plateau_mask = (t >= plateau_start) & (t <= plateau_end)
    plateau_slope = line_slope(t[plateau_mask], v[plateau_mask]) if plateau_mask.sum() > 5 else np.nan

    plateau_level_mask = (t >= offset_s - 1.0) & (t <= offset_s)
    if plateau_level_mask.sum():
        plateau_level = float(np.median(v[plateau_level_mask]))
    else:
        plateau_level = float(np.median(v[plateau_mask]))

    min_rel_idx = int(np.argmin(post_v))
    min_t = float(post_t[min_rel_idx])
    min_v = float(post_v[min_rel_idx])

    undershoot_signed = min_v - baseline
    undershoot_mag = max(0.0, baseline - min_v)

    drop = plateau_level - min_v
    offset_idx = np.searchsorted(t, offset_s)
    min_idx = np.searchsorted(t, min_t)

    thr80d = plateau_level - 0.2 * drop
    thr63d = plateau_level - 0.632 * drop
    thr20d = plateau_level - 0.8 * drop

    td80 = first_crossing_time(t, v, thr80d, offset_idx, min_idx, direction="down") if drop > 0 else np.nan
    td63 = first_crossing_time(t, v, thr63d, offset_idx, min_idx, direction="down") if drop > 0 else np.nan
    td20 = first_crossing_time(t, v, thr20d, offset_idx, min_idx, direction="down") if drop > 0 else np.nan

    decay_slope = np.nan
    if np.isfinite(td80) and np.isfinite(td20) and td20 > td80:
        decay_slope = (0.6 * drop) / (td20 - td80)

    decay_tau = td63 - offset_s if np.isfinite(td63) else np.nan

    # Return slope block: this matches the code path that generated the threshold CSV.
    v_smooth = moving_average(v_raw, max(5, int(round(smoothing_window_s / dt))))
    min_idx_raw = int(np.argmin(np.abs(t - min_t)))
    min_v_smooth = float(v_smooth[min_idx_raw])

    final_t = float(t[-1])
    if final_value_mode == "last_200ms_mean":
        end_mask = t >= (t[-1] - 0.2)
        final_v = float(np.mean(v_raw[end_mask]))
    else:
        final_v = float(v_raw[-1])

    if final_t > min_t:
        return_slope = (final_v - min_v_smooth) / (final_t - min_t)
    else:
        return_slope = np.nan

    return {
        "stim_onset_s": float(onset_s),
        "stim_offset_s": float(offset_s),
        "baseline_mV": baseline,
        "peak_t_s": peak_t,
        "peak_mV": peak_v,
        "peak_depolarization_mV": peak_dep,
        "rise_slope_mV_per_s": rise_slope,
        "rise_tau_s": rise_tau,
        "plateau_level_mV": plateau_level,
        "plateau_slope_mV_per_s": plateau_slope,
        "undershoot_min_t_s": min_t,
        "undershoot_min_mV": min_v,
        "undershoot_signed_mV": undershoot_signed,
        "undershoot_magnitude_mV": undershoot_mag,
        "decay_slope_mV_per_s": decay_slope,
        "decay_tau_s": decay_tau,
        "final_t_s": final_t,
        "final_mV": final_v,
        "return_slope_mV_per_s": return_slope,
        "final_minus_baseline_mV": final_v - baseline,
    }


def describe_values(x: Sequence[float]) -> Dict[str, float]:
    x = pd.Series(x).dropna().astype(float)
    n = len(x)
    if n == 0:
        return {
            "n": 0, "mean": np.nan, "median": np.nan, "std": np.nan, "sem": np.nan,
            "min": np.nan, "q1": np.nan, "q3": np.nan, "max": np.nan, "iqr": np.nan,
            "ci95_low": np.nan, "ci95_high": np.nan,
        }

    if n == 1:
        ci_low = ci_high = float(x.iloc[0])
        sem = np.nan
        std = np.nan
    else:
        mean = float(x.mean())
        sem = float(stats.sem(x, nan_policy="omit"))
        ci_low, ci_high = stats.t.interval(0.95, df=n - 1, loc=mean, scale=sem)
        std = float(x.std(ddof=1))

    return {
        "n": int(n),
        "mean": float(x.mean()),
        "median": float(x.median()),
        "std": std,
        "sem": sem if n > 1 else np.nan,
        "min": float(x.min()),
        "q1": float(x.quantile(0.25)),
        "q3": float(x.quantile(0.75)),
        "max": float(x.max()),
        "iqr": float(x.quantile(0.75) - x.quantile(0.25)),
        "ci95_low": float(ci_low),
        "ci95_high": float(ci_high),
    }

In [ ]:
# =========================
# Threshold selection
# =========================
def get_threshold_band(threshold_df: pd.DataFrame, sweep_id: int, feature: str,
                       group: str, mode: str) -> Tuple[float, float]:
    row = threshold_df[
        (threshold_df["group"] == group)
        & (threshold_df["sweep"] == sweep_id)
        & (threshold_df["feature"] == feature)
    ]
    if row.empty:
        raise ValueError(f"No threshold row found for group={group}, sweep={sweep_id}, feature={feature}")
    row = row.iloc[0]

    if mode == "min_max":
        return float(row["min"]), float(row["max"])
    if mode == "q1_q3":
        return float(row["acceptable_low_q1"]), float(row["acceptable_high_q3"])
    if mode == "ci95":
        return float(row["acceptable_low_ci95"]), float(row["acceptable_high_ci95"])
    raise ValueError(f"Unknown THRESHOLD_MODE: {mode}")


def soften_threshold_band(low: float, high: float,
                          pad_frac: float = 0.0,
                          pad_abs: float = 0.0) -> Tuple[float, float]:
    low = float(low)
    high = float(high)
    width = max(high - low, 0.0)
    pad = max(float(pad_abs), float(pad_frac) * width)
    return low - pad, high + pad


# =========================
# Trial extraction
# =========================
def build_trial_table(study, fixed_params: Dict[str, object], penalty_value: float,
                      include_penalty_trials: bool) -> pd.DataFrame:
    rows = []
    all_param_keys = list(ORDERED_PARAM_KEYS)

    # add any other keys seen in the study
    seen_extra = set()
    for t in study.trials:
        seen_extra.update(t.params.keys())
    for k in sorted(seen_extra):
        if k not in all_param_keys:
            all_param_keys.append(k)

    for t in study.trials:
        if t.state != optuna.trial.TrialState.COMPLETE:
            continue

        is_penalty = (t.value is None) or (float(t.value) >= penalty_value)
        if is_penalty and not include_penalty_trials:
            continue

        full = dict(fixed_params)
        full.update(normalize_flat_params(t.params))

        row = {
            "trial_number": int(t.number),
            "objective": float(t.value) if t.value is not None else np.nan,
            "is_penalty": bool(is_penalty),
            "state": str(t.state),
        }
        for k in all_param_keys:
            row[k] = full.get(k, np.nan)

        rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df = df.sort_values(["objective", "trial_number"], ascending=[True, True]).reset_index(drop=True)
    return df


# =========================
# Simulation
# =========================
def simulate_trial_aligned(
    full_params: Dict[str, object],
    context: ExperimentContext,
    objective_loss_type: str = "COMBINED",
) -> Dict[str, object]:
    paramdict = build_paramdict(context.experiment_type, context.current_na, full_params)

    with warnings.catch_warnings():
        warnings.simplefilter("error", RuntimeWarning)
        z = odeint(model, Z0, context.sim_time_ms, args=(paramdict,))

    if not np.isfinite(z).all():
        raise ValueError("Non-finite values detected in ODE solution")

    vm_raw = np.asarray(z[:, 0], dtype=float)
    vm_stable = vm_raw[context.stable_index_simulation:]
    t_stable_ms = context.sim_time_ms_stable

    vm_interp = np.interp(context.exp_times_ms_stable, t_stable_ms, vm_stable)
    vm_interp_norm = normalize_trace_for_target_mode(vm_interp, context.target_mean_mode)

    vm_downsampled = downsample_array_median_fast(vm_stable, len(context.exp_trace_stable))
    vm_downsampled_norm = normalize_trace_for_target_mode(vm_downsampled, context.target_mean_mode)

    objective_recomputed = compute_loss(vm_downsampled_norm, context.exp_trace_stable, loss_type=objective_loss_type)
    features = extract_features_from_trace(
        context.exp_times_ms_stable / 1000.0,
        vm_interp_norm,
        onset_s=context.feature_onset_s,
        offset_s=context.feature_offset_s,
    )

    return {
        "paramdict": paramdict,
        "z": z,
        "vm_stable_raw": vm_stable,
        "vm_interp": vm_interp_norm,
        "vm_downsampled": vm_downsampled_norm,
        "objective_recomputed": objective_recomputed,
        "features": features,
    }

In [ ]:
# =========================
# Plotting helpers
# =========================
def plot_top_trial_overlay(
    context: ExperimentContext,
    simulations: Dict[int, Dict[str, object]],
    trial_df: pd.DataFrame,
    save_path: Path,
    accepted_trial_numbers: Optional[Sequence[int]] = None,
) -> None:
    accepted_set = set(accepted_trial_numbers or [])

    plt.figure(figsize=(11, 5.5))
    for _, row in trial_df.iterrows():
        trial_number = int(row["trial_number"])
        if trial_number not in simulations:
            continue
        y = simulations[trial_number]["vm_interp"]

        if trial_number in accepted_set:
            plt.plot(
                context.exp_times_ms_stable / 1000.0, y,
                alpha=TRACE_ALPHA_ACCEPTED, linewidth=1.0,
                label="accepted baseline" if trial_number == min(accepted_set) else None,
            )
        else:
            plt.plot(
                context.exp_times_ms_stable / 1000.0, y,
                alpha=TRACE_ALPHA_TOP, linewidth=0.9,
                label="top valid trial" if trial_number == int(trial_df.iloc[0]["trial_number"]) else None,
            )

    # highlight best trial
    if len(trial_df):
        best_trial_num = int(trial_df.iloc[0]["trial_number"])
        if best_trial_num in simulations:
            plt.plot(
                context.exp_times_ms_stable / 1000.0, simulations[best_trial_num]["vm_interp"],
                linewidth=BEST_TRACE_LINEWIDTH, label=f"best trial #{best_trial_num}",
            )

    plt.plot(
        context.exp_times_ms_stable / 1000.0, context.exp_trace_stable,
        linewidth=EXPERIMENT_LINEWIDTH, linestyle="--", label="experimental trace",
    )
    plt.axvline(context.feature_onset_s, linestyle="--", linewidth=1.0)
    plt.axvline(context.feature_offset_s, linestyle="--", linewidth=1.0)
    plt.xlabel("Time (s)")
    plt.ylabel("Vm (centered mV)" if context.target_mean_mode != "default" else "Vm (mV)")
    plt.title(
        f"{context.experiment_type} {context.current_na} nA — top trials vs experimental trace"
    )
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.close()


def plot_feature_boxplots(
    top_features_df: pd.DataFrame,
    accepted_features_df: pd.DataFrame,
    threshold_df: pd.DataFrame,
    sweep_id: int,
    group: str,
    mode: str,
    out_dir: Path,
) -> None:
    for feature in FEATURES_FOR_FILTER:
        plt.figure(figsize=(6.5, 4.8))

        data = [top_features_df[feature].dropna().to_numpy()]
        labels = ["top_valid"]

        if not accepted_features_df.empty:
            data.append(accepted_features_df[feature].dropna().to_numpy())
            labels.append("accepted")

        plt.boxplot(data, tick_labels=labels, showfliers=False)

        rng = np.random.default_rng(0)
        for i, vals in enumerate(data, start=1):
            if len(vals) == 0:
                continue
            xj = rng.normal(loc=i, scale=0.04, size=len(vals))
            plt.plot(xj, vals, "o", alpha=0.6, markersize=4)

        low, high = get_threshold_band(threshold_df, sweep_id, feature, group, mode)
        plt.axhspan(low, high, alpha=0.15)
        plt.ylabel(feature)
        plt.title(f"{feature} — threshold mode: {mode}, group: {group}")
        plt.tight_layout()
        plt.savefig(out_dir / f"{feature}_boxplot.png", dpi=180, bbox_inches="tight")
        plt.close()


def plot_parameter_sweep_deltas(sweep_df: pd.DataFrame, out_dir: Path) -> None:
    if sweep_df.empty:
        return

    for param_key in sorted(sweep_df["parameter"].dropna().unique()):
        g_param = sweep_df[sweep_df["parameter"] == param_key].copy()

        for feature in FEATURES_FOR_FILTER:
            delta_col = f"delta_{feature}"
            if delta_col not in g_param.columns:
                continue

            valid_param = g_param[np.isfinite(g_param[delta_col])].copy()
            if valid_param.empty:
                continue

            plt.figure(figsize=(7.2, 4.8))

            for baseline_id, gb in valid_param.groupby("baseline_trial_number"):
                gb = gb.sort_values("fold")
                plt.plot(gb["fold"], gb[delta_col], alpha=0.25, linewidth=1.0)

            summary = (
                valid_param.groupby("fold", as_index=False)[delta_col]
                .agg(
                    median="median",
                    q25=lambda x: np.nanquantile(x, 0.25),
                    q75=lambda x: np.nanquantile(x, 0.75),
                )
                .sort_values("fold")
            )

            plt.plot(summary["fold"], summary["median"], linewidth=2.0, label="median")
            plt.fill_between(summary["fold"], summary["q25"], summary["q75"], alpha=0.20, label="IQR")

            plt.axhline(0.0, linestyle="--", linewidth=1.0)
            plt.xlabel(f"{param_key} fold-change")
            plt.ylabel(f"Δ {feature}")
            plt.title(f"{param_key}: change in {feature} relative to baseline")
            plt.legend(loc="best")
            plt.tight_layout()
            plt.savefig(out_dir / f"{param_key}__{feature}__delta.png", dpi=180, bbox_inches="tight")
            plt.close()

In [ ]:
# =========================
# Summary tables
# =========================
def summarize_feature_table(df: pd.DataFrame, set_name: str) -> pd.DataFrame:
    rows = []
    for feature in FEATURES_FOR_FILTER:
        if feature not in df.columns:
            continue
        d = describe_values(df[feature])
        d["feature"] = feature
        d["set_name"] = set_name
        rows.append(d)
    return pd.DataFrame(rows)


def build_feature_pass_table(
    features_df: pd.DataFrame,
    threshold_df: pd.DataFrame,
    sweep_id: int,
    group: str,
    mode: str,
) -> pd.DataFrame:
    out = features_df.copy()
    pass_cols = []
    active_features = [f for f in FEATURES_FOR_FILTER if f in out.columns]

    for feature in active_features:
        raw_low, raw_high = get_threshold_band(threshold_df, sweep_id, feature, group, mode)
        low, high = soften_threshold_band(
            raw_low,
            raw_high,
            pad_frac=SOFT_THRESHOLD_PAD_FRAC,
            pad_abs=SOFT_THRESHOLD_PAD_ABS,
        )
        col = f"pass_{feature}"
        pass_cols.append(col)
        out[col] = out[feature].apply(lambda x: np.isfinite(x) and (x >= low) and (x <= high))
        out[f"threshold_low_raw_{feature}"] = raw_low
        out[f"threshold_high_raw_{feature}"] = raw_high
        out[f"threshold_low_{feature}"] = low
        out[f"threshold_high_{feature}"] = high

    if not pass_cols:
        out["n_features_checked"] = 0
        out["n_features_passed"] = 0
        out["pass_fraction"] = np.nan
        out["n_features_failed"] = 0
        out["good_enough"] = False
        return out

    out["n_features_checked"] = len(pass_cols)
    out["n_features_passed"] = out[pass_cols].sum(axis=1)
    out["pass_fraction"] = out["n_features_passed"] / float(len(pass_cols))
    out["n_features_failed"] = out["n_features_checked"] - out["n_features_passed"]

    if REQUIRE_ALL_FEATURES_IN_RANGE:
        out["good_enough"] = out[pass_cols].all(axis=1)
        return out

    if FEATURE_ACCEPTANCE_MODE == "all":
        out["good_enough"] = out[pass_cols].all(axis=1)
    elif FEATURE_ACCEPTANCE_MODE == "any":
        out["good_enough"] = out[pass_cols].any(axis=1)
    elif FEATURE_ACCEPTANCE_MODE == "fraction":
        out["good_enough"] = out["pass_fraction"] >= float(MIN_PASS_FRACTION)
    elif FEATURE_ACCEPTANCE_MODE == "max_failed":
        out["good_enough"] = out["n_features_failed"] <= int(MAX_FAILED_FEATURES)
    else:
        raise ValueError(f"Unknown FEATURE_ACCEPTANCE_MODE: {FEATURE_ACCEPTANCE_MODE}")

    return out


# =========================
# Parameter sweeps
# =========================
def select_accepted_for_sweeps(accepted_df: pd.DataFrame, max_accepted: Optional[int]) -> pd.DataFrame:
    if max_accepted is None or len(accepted_df) <= max_accepted:
        return accepted_df.copy()
    return accepted_df.sort_values(["objective", "trial_number"]).head(max_accepted).copy()


def run_parameter_sweeps(
    accepted_df: pd.DataFrame,
    simulation_cache: Dict[int, Dict[str, object]],
    context: ExperimentContext,
    param_sweep_folds: Dict[str, Sequence[float]],
) -> pd.DataFrame:
    if accepted_df.empty:
        return pd.DataFrame()

    records = []

    for _, row in accepted_df.iterrows():
        baseline_trial_number = int(row["trial_number"])
        baseline_features = simulation_cache[baseline_trial_number]["features"]
        baseline_params = {k: row[k] for k in ORDERED_PARAM_KEYS if k in row.index}

        for param_key, folds in param_sweep_folds.items():
            base_val = baseline_params.get(param_key, np.nan)
            if pd.isna(base_val):
                continue

            # Hill-mode baselines do not use zth/zs in the same way.
            if baseline_params.get("switching_function", "sigmoid") == "hill" and param_key in {"zth", "zs"}:
                continue

            for fold in folds:
                perturbed_params = dict(baseline_params)

                if fold == 1.0:
                    feat = baseline_features
                    perturbed_value = base_val
                else:
                    perturbed_value = float(base_val) * float(fold)

                    # keep positive parameters positive
                    if param_key in {"gki", "d", "pk", "zth", "zs", "gs"}:
                        perturbed_value = max(1e-12, perturbed_value)

                    perturbed_params[param_key] = perturbed_value

                    try:
                        sim = simulate_trial_aligned(perturbed_params, context)
                        feat = sim["features"]
                    except Exception as exc:
                        record = {
                            "baseline_trial_number": baseline_trial_number,
                            "parameter": param_key,
                            "fold": float(fold),
                            "baseline_value": float(base_val),
                            "perturbed_value": np.nan,
                            "simulation_ok": False,
                            "error": str(exc),
                        }
                        for feature in FEATURES_FOR_FILTER:
                            record[feature] = np.nan
                            record[f"baseline_{feature}"] = baseline_features.get(feature, np.nan)
                            record[f"delta_{feature}"] = np.nan
                        records.append(record)
                        continue

                record = {
                    "baseline_trial_number": baseline_trial_number,
                    "parameter": param_key,
                    "fold": float(fold),
                    "baseline_value": float(base_val),
                    "perturbed_value": float(perturbed_value),
                    "simulation_ok": True,
                    "error": "",
                }
                for feature in FEATURES_FOR_FILTER:
                    record[feature] = feat.get(feature, np.nan)
                    record[f"baseline_{feature}"] = baseline_features.get(feature, np.nan)
                    if np.isfinite(record[feature]) and np.isfinite(record[f"baseline_{feature}"]):
                        record[f"delta_{feature}"] = record[feature] - record[f"baseline_{feature}"]
                    else:
                        record[f"delta_{feature}"] = np.nan
                records.append(record)

    return pd.DataFrame(records)

In [ ]:
cfg = RunConfig(
    db_path=Path(DB_PATH),
    atf_path=Path(ATF_PATH),
    threshold_csv_path=Path(THRESHOLD_CSV_PATH),
    out_dir=Path(OUT_DIR),
    study_name=STUDY_NAME,
    experiment_type=EXPERIMENT_TYPE,
    current_na=CURRENT_NA,
    target_mean_mode=TARGET_MEAN_MODE,
)

out_paths = make_output_dirs(cfg.out_dir)

if not cfg.db_path.exists():
    raise FileNotFoundError(cfg.db_path)
if not cfg.atf_path.exists():
    raise FileNotFoundError(cfg.atf_path)
if not cfg.threshold_csv_path.exists():
    raise FileNotFoundError(cfg.threshold_csv_path)

study_name = cfg.study_name or infer_study_name_from_db(cfg.db_path)
inferred_experiment_type, inferred_current_na, inferred_target_mean_mode = infer_metadata_from_study_name(study_name)

experiment_type = cfg.experiment_type or inferred_experiment_type
current_na = cfg.current_na or inferred_current_na
target_mean_mode = cfg.target_mean_mode or inferred_target_mean_mode

if target_mean_mode == "centered_scaled":
    print(
        "Warning: target_mean_mode == 'centered_scaled'. "
        "Feature thresholds in mV are usually not comparable to scaled traces."
    )

print(f"Study name: {study_name}")
print(f"Experiment type: {experiment_type}")
print(f"Current (nA): {current_na}")
print(f"Target mean mode: {target_mean_mode}")

study = load_study(cfg.db_path, study_name=study_name)
fixed_params = load_fixed_params_from_db(cfg.db_path)
threshold_df = pd.read_csv(cfg.threshold_csv_path)

exp_times_ms_stable, exp_trace_stable = load_experimental_trace(
    cfg.atf_path, experiment_type, current_na, target_mean_mode
)

last_trace_time_value_ms = float(exp_times_ms_stable[-1])
sim_time_ms = np.linspace(0.0, last_trace_time_value_ms, int(last_trace_time_value_ms / SIM_DT_MS))
stable_index_simulation = int(round(len(sim_time_ms) * EXPERIMENT_CUTS[experiment_type][1]))
if stable_index_simulation >= len(sim_time_ms):
    raise ValueError("stable_index_simulation is beyond the array length")

exp_features = extract_features_from_trace(exp_times_ms_stable / 1000.0, exp_trace_stable)
feature_onset_s = float(exp_features["stim_onset_s"])
feature_offset_s = float(exp_features["stim_offset_s"])

context = ExperimentContext(
    experiment_type=experiment_type,
    current_na=current_na,
    target_mean_mode=target_mean_mode,
    study_name=study_name,
    current_column_id=CURRENT_DICT_COLUMNS[str(current_na)],
    sweep_id=CURRENT_DICT_COLUMNS[str(current_na)],
    cut_after_index=EXPERIMENT_CUTS[experiment_type][0],
    cut_before_ratio=EXPERIMENT_CUTS[experiment_type][1],
    exp_times_ms_stable=exp_times_ms_stable,
    exp_trace_stable=exp_trace_stable,
    sim_time_ms=sim_time_ms,
    sim_time_ms_stable=sim_time_ms[stable_index_simulation:],
    stable_index_simulation=stable_index_simulation,
    fixed_params=fixed_params,
    threshold_df=threshold_df,
    feature_onset_s=feature_onset_s,
    feature_offset_s=feature_offset_s,
)

Study name: CONTROL_175nA_centered_L2_2500t
Experiment type: CONTROL
Current (nA): 175
Target mean mode: centered


In [ ]:
trial_df_all = build_trial_table(
    study,
    fixed_params=fixed_params,
    penalty_value=PENALTY_VALUE,
    include_penalty_trials=INCLUDE_PENALTY_TRIALS,
)
if trial_df_all.empty:
    raise RuntimeError("No completed trials found after penalty filtering.")

top_trial_df = trial_df_all.head(min(TOP_N_REQUESTED, len(trial_df_all))).copy()
print(f"Using {len(top_trial_df)} completed trials for simulation/inspection.")

Using 300 completed trials for simulation/inspection.


In [ ]:
simulation_cache: Dict[int, Dict[str, object]] = {}
feature_rows = []
failed_trials = []

for _, row in top_trial_df.iterrows():
    trial_number = int(row["trial_number"])
    full_params = {k: row[k] for k in ORDERED_PARAM_KEYS if k in row.index}
    try:
        sim = simulate_trial_aligned(full_params, context)
        simulation_cache[trial_number] = sim

        feat_row = {
            "trial_number": trial_number,
            "objective": float(row["objective"]),
            "objective_recomputed": float(sim["objective_recomputed"]),
        }
        for k in ORDERED_PARAM_KEYS:
            if k in row.index:
                feat_row[k] = row[k]
        feat_row.update(sim["features"])
        feature_rows.append(feat_row)
    except Exception as exc:
        failed_trials.append({"trial_number": trial_number, "error": str(exc)})
        print(f"Trial {trial_number} failed during re-simulation: {exc}")

if not feature_rows:
    raise RuntimeError("All selected trials failed during re-simulation.")

top_features_df = pd.DataFrame(feature_rows).sort_values(["objective", "trial_number"]).reset_index(drop=True)
failed_df = pd.DataFrame(failed_trials)

Trial 1103 failed during re-simulation: overflow encountered in exp


In [ ]:
feature_pass_df = build_feature_pass_table(
    top_features_df, threshold_df, context.sweep_id, THRESHOLD_GROUP, THRESHOLD_MODE
)
accepted_df = feature_pass_df[feature_pass_df["good_enough"]].copy()
accepted_df = accepted_df.sort_values(["objective", "trial_number"]).reset_index(drop=True)

trial_df_all.to_csv(out_paths["tables"] / "all_completed_trials_after_penalty_filter.csv", index=False)
top_trial_df.to_csv(out_paths["tables"] / "top_trial_param_values.csv", index=False)
top_features_df.to_csv(out_paths["tables"] / "top_trial_features.csv", index=False)
feature_pass_df.to_csv(out_paths["tables"] / "top_trial_features_with_threshold_pass_fail.csv", index=False)
accepted_df.to_csv(out_paths["tables"] / "accepted_good_enough_baselines.csv", index=False)
if not failed_df.empty:
    failed_df.to_csv(out_paths["tables"] / "failed_resimulations.csv", index=False)

top_summary_df = summarize_feature_table(top_features_df, "top_valid")
accepted_summary_df = summarize_feature_table(accepted_df, "accepted") if not accepted_df.empty else pd.DataFrame()
experimental_feature_df = pd.DataFrame([exp_features])

top_summary_df.to_csv(out_paths["tables"] / "top_trial_feature_summary.csv", index=False)
if not accepted_summary_df.empty:
    accepted_summary_df.to_csv(out_paths["tables"] / "accepted_baseline_feature_summary.csv", index=False)
experimental_feature_df.to_csv(out_paths["tables"] / "experimental_trace_features.csv", index=False)

In [ ]:
plot_top_trial_overlay(
    context,
    simulations=simulation_cache,
    trial_df=top_features_df[["trial_number", "objective"]],
    save_path=out_paths["plots"] / "top_trials_overlay_interpolated.png",
    accepted_trial_numbers=accepted_df["trial_number"].tolist(),
)

accepted_trial_numbers = accepted_df["trial_number"].tolist()
if accepted_trial_numbers:
    accepted_trial_df = top_features_df[top_features_df["trial_number"].isin(accepted_trial_numbers)].copy()
    plot_top_trial_overlay(
        context,
        simulations=simulation_cache,
        trial_df=accepted_trial_df[["trial_number", "objective"]],
        save_path=out_paths["plots"] / "accepted_baselines_overlay_interpolated.png",
        accepted_trial_numbers=accepted_trial_numbers,
    )

plot_feature_boxplots(
    top_features_df=top_features_df,
    accepted_features_df=accepted_df,
    threshold_df=threshold_df,
    sweep_id=context.sweep_id,
    group=THRESHOLD_GROUP,
    mode=THRESHOLD_MODE,
    out_dir=out_paths["plots"],
)

In [ ]:
accepted_for_sweeps_df = select_accepted_for_sweeps(accepted_df, MAX_ACCEPTED_FOR_SWEEPS)
parameter_sweep_df = run_parameter_sweeps(
    accepted_df=accepted_for_sweeps_df,
    simulation_cache=simulation_cache,
    context=context,
    param_sweep_folds=PARAM_SWEEP_FOLDS,
)
parameter_sweep_df.to_csv(out_paths["tables"] / "parameter_sweep_feature_deltas.csv", index=False)
plot_parameter_sweep_deltas(parameter_sweep_df, out_paths["sweep_plots"])

In [ ]:
summary_lines = []
summary_lines.append(f"study_name: {study_name}")
summary_lines.append(f"experiment_type: {experiment_type}")
summary_lines.append(f"current_na: {current_na}")
summary_lines.append(f"target_mean_mode: {target_mean_mode}")
summary_lines.append(f"threshold_group: {THRESHOLD_GROUP}")
summary_lines.append(f"threshold_mode: {THRESHOLD_MODE}")
summary_lines.append(f"feature_acceptance_mode: {FEATURE_ACCEPTANCE_MODE}")
summary_lines.append(f"min_pass_fraction: {MIN_PASS_FRACTION}")
summary_lines.append(f"max_failed_features: {MAX_FAILED_FEATURES}")
summary_lines.append(f"soft_threshold_pad_frac: {SOFT_THRESHOLD_PAD_FRAC}")
summary_lines.append(f"soft_threshold_pad_abs: {SOFT_THRESHOLD_PAD_ABS}")
summary_lines.append(f"top_n_requested: {TOP_N_REQUESTED}")
summary_lines.append(f"top_n_used: {len(top_trial_df)}")
summary_lines.append(f"resimulated_successfully: {len(top_features_df)}")
summary_lines.append(f"accepted_good_enough: {len(accepted_df)}")
summary_lines.append(f"accepted_used_for_sweeps: {len(accepted_for_sweeps_df)}")
summary_lines.append(f"parameter_sweep_rows: {len(parameter_sweep_df)}")
summary_lines.append(f"experimental_onset_s: {feature_onset_s:.4f}")
summary_lines.append(f"experimental_offset_s: {feature_offset_s:.4f}")

summary_lines.append(
    "note: in the model, the gap term enters as P_kgap = d_gap * P_k, so one-at-a-time sweeps of d and pk are not independent mechanisms."
)

summary_text = "\n".join(summary_lines)
print("\n" + summary_text)

with open(out_paths["base"] / "run_summary.txt", "w") as f:
    f.write(summary_text + "\n")

print(f"\nOutputs written to: {out_paths['base']}")


study_name: CONTROL_175nA_centered_L2_2500t
experiment_type: CONTROL
current_na: 175
target_mean_mode: centered
threshold_group: pooled
threshold_mode: min_max
feature_acceptance_mode: fraction
min_pass_fraction: 0.75
max_failed_features: 2
soft_threshold_pad_frac: 0.1
soft_threshold_pad_abs: 0.0
top_n_requested: 300
top_n_used: 300
resimulated_successfully: 299
accepted_good_enough: 87
accepted_used_for_sweeps: 87
parameter_sweep_rows: 2958
experimental_onset_s: 11.1730
experimental_offset_s: 31.1730
note: in the model, the gap term enters as P_kgap = d_gap * P_k, so one-at-a-time sweeps of d and pk are not independent mechanisms.

Outputs written to: /content/astro_post_optuna_outputs


In [ ]:
import os
import shutil
from google.colab import files

# Path to the folder you want to download
output_folder = "/content/astro_post_optuna_outputs"   # change this to your actual folder

# Name of the zip file to create
zip_base = "/content/astro_post_optuna_outputs_download"  # zip will be output_download.zip

# Create zip
shutil.make_archive(zip_base, "zip", output_folder)

# Download zip to your computer
files.download(zip_base + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>